In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.bronze")

In [0]:
bucket = "s3://ro-company-lake"

onrc_firme_path = f"{bucket}/raw_v2/onrc/firme/"
onrc_nomenclatoare_path = f"{bucket}/raw_v2/onrc/nomenclatoare/"

display(dbutils.fs.ls(onrc_firme_path))
display(dbutils.fs.ls(onrc_nomenclatoare_path))

In [0]:
od_stare_firma = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "^")
    .option("encoding", "UTF-8")
    .csv(f"{onrc_firme_path}snapshot_date=*/package=*/od_stare_firma.csv")
)

display(od_stare_firma.limit(20))
print("Rows:", od_stare_firma.count())
print("Columns:", od_stare_firma.columns)

In [0]:
(
    od_stare_firma
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.bronze.onrc_stare_firma_raw")
)

In [0]:
n_stare_firma = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "^")
    .option("encoding", "UTF-8")
    .csv(f"{onrc_nomenclatoare_path}snapshot_date=*/package=*/n_stare_firma.csv")
)

display(n_stare_firma.limit(20))
print("Rows:", n_stare_firma.count())
print("Columns:", n_stare_firma.columns)

In [0]:
(
    n_stare_firma
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.bronze.n_stare_firma_raw")
)

In [0]:
display(spark.sql("""
SELECT 'onrc_stare_firma_raw' AS table_name, COUNT(*) AS rows
FROM company_ro.bronze.onrc_stare_firma_raw

UNION ALL

SELECT 'n_stare_firma_raw', COUNT(*)
FROM company_ro.bronze.n_stare_firma_raw
"""))